##### Step 1: Create the carrier/route performance Metric View

In [0]:
%sql
CREATE OR REPLACE VIEW airline_analytics.gold.mv_carrier_route_performance
WITH METRICS
LANGUAGE YAML
COMMENT 'Semantic layer over agg_daily_carrier_route_performance for AI/BI dashboards and future Genie use'
AS $$

version: '1.1'
source: airline_analytics.gold.agg_daily_carrier_route_performance
joins:
  - name: carrier
    source: airline_analytics.gold.dim_carrier
    on: source.carrier_code = carrier.carrier_code AND source.flight_date BETWEEN carrier.effective_start AND carrier.effective_end
    cardinality: many_to_one


dimensions:
  - name: flight_date
    expr: flight_date
  - name: year
    expr: year(flight_date)
  - name: quarter
    expr: quarter(flight_date)
  - name: month
    expr: month(flight_date)
  - name: carrier_code
    expr: carrier_code
  - name: origin
    expr: origin
  - name: dest
    expr: dest
  - name: route
    expr: route

measures:
  - name: total_flights
    expr: SUM(flight_count)
  - name: completed_flights
    expr: SUM(completed_count)
  - name: cancelled_flights
    expr: SUM(cancelled_count)
  - name: diverted_flights
    expr: SUM(diverted_count)
  - name: cancellation_rate
    expr: SUM(cancelled_count) * 1.0 / SUM(flight_count)
  - name: on_time_dep_pct
    expr: 1 - (SUM(dep_del15_count) * 1.0 / SUM(completed_count))
  - name: on_time_arr_pct
    expr: 1 - (SUM(arr_del15_count) * 1.0 / SUM(completed_count))
  - name: avg_dep_delay_min
    expr: SUM(avg_dep_delay_min * completed_count) / SUM(completed_count)
  - name: avg_arr_delay_min
    expr: SUM(avg_arr_delay_min * completed_count) / SUM(completed_count)
  - name: avg_taxi_out_min
    expr: SUM(avg_taxi_out_min * completed_count) / SUM(completed_count)
  - name: avg_taxi_in_min
    expr: SUM(avg_taxi_in_min * completed_count) / SUM(completed_count)
  - name: total_distance_mi
    expr: SUM(total_distance_mi)
  - name: carrier_delay_min
    expr: SUM(carrier_delay_min)
  - name: weather_delay_min
    expr: SUM(weather_delay_min)
  - name: nas_delay_min
    expr: SUM(nas_delay_min)
  - name: security_delay_min
    expr: SUM(security_delay_min)
  - name: late_aircraft_delay_min
    expr: SUM(late_aircraft_delay_min)
$$

In [0]:
%sql
select carrier_code, MEASURE(total_flights) as total_flights
from airline_analytics.gold.mv_carrier_route_performance
group by carrier_code order by total_flights desc limit 10